# 第13回　多変量解析への招待：主成分分析（PCA）
## ―― たくさんの変数の背後にある、少数の軸を見つける

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> 変数がたくさんあっても、**本質はもっと少ない次元**で表せることが多い。

### フック

> 北辰大データには、勉強・睡眠・出席・朝食・SNS… とたくさんの変数がある。
> **全部の関係を一度に見るのは無理**（変数5個でも散布図は10通り、10個なら45通り）。
>
> これらの背後にある「**本当の軸**」は、いくつあるのだろう？

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
df = df.dropna().reset_index(drop=True)
print("準備OK")

---
## 1. 主成分分析（PCA）の発想

生活に関する5つの変数を使う：**勉強時間・睡眠時間・出席率・朝食日数・SNS時間**。

PCAは、これら5次元のデータを、**ばらつき（情報）が最も大きい方向**に新しい軸を引き直す。最初の軸（第1主成分）が一番多くの情報を持ち、次の軸（第2主成分）がその次…と続く。少数の軸だけ残せば**次元を削減**できる。

（注意：変数ごとに単位が違う（時間・%・日）ので、まず**標準化**して土俵をそろえる。）

In [ ]:
cols = ["勉強時間h", "睡眠時間h", "出席率", "朝食日数week", "SNS時間h"]
X = StandardScaler().fit_transform(df[cols])   # 標準化
pca = PCA().fit(X)

print("寄与率（各主成分が説明する情報の割合）")
cum = 0
for i, r in enumerate(pca.explained_variance_ratio_):
    cum += r
    print(f"  第{i+1}主成分: {r*100:4.1f}%   （累積 {cum*100:4.1f}%）")

**第1主成分だけで全体の約61%**、第2主成分まで合わせると約74%の情報を説明できる。5次元のデータの本質は、ほぼ**2次元**に圧縮できるということだ。

では、その第1主成分は「何の軸」なのか？　各変数がどれくらい効いているか（**負荷**）を見て、人間が**命名**する。

In [ ]:
負荷 = pd.DataFrame(pca.components_[:2].T, index=cols, columns=["第1主成分", "第2主成分"])
print(負荷.round(2))
print("\n第1主成分：勉強(+)・睡眠(+)・出席(+)・朝食(+) が同じ向き、SNS(−) が逆向き。")
print("→ 『きちんとした生活をしている度合い』＝《生活の規律》の軸、と命名できる。")

第1主成分は、勉強・睡眠・出席・朝食が高くSNSが低い人ほど大きくなる。これは「**生活がきちんとしている度合い（規律）**」と読める。

PCAは、もともと別々だった5つの変数の背後に、**1本の“規律”という軸**が隠れていたことを浮かび上がらせた。これが**次元削減＝本質の抽出**だ。

---
## 2. 2次元に圧縮して可視化

各学生を、第1主成分（規律）と第2主成分の2軸で配置する。色をテスト点にすると、規律の高い（右側の）人ほど点が高い傾向が見える。

In [ ]:
Z = PCA(n_components=2).fit_transform(X)    # 2次元に圧縮した座標
plt.figure(figsize=(7, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=df["テスト点"], cmap="viridis", s=18, alpha=0.8)
plt.colorbar(sc, label="テスト点")
plt.xlabel("第1主成分（生活の規律 →）"); plt.ylabel("第2主成分")
plt.title("5次元を2次元に圧縮（情報の約74%を保持）")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.show()
print("右へ行くほど（規律が高いほど）テスト点が高い色になりやすい。")

---
## 3. 大事な注意

- **PCAは情報を失う**。今回、第1・第2主成分で説明できたのは約**74%**。残りの**約26%は捨てている**。圧縮はタダではない。
- **主成分に必ず意味があるとは限らない**。今回はたまたま「規律」と解釈できたが、現実のデータでは「何だかよく分からない軸」も多い。**意味づけ（命名）は人間の解釈**であり、数学が保証してくれるものではない。
- 主成分の**向き（符号）は反転することがある**（右が規律高でも左が規律高でも数学的には同じ）。

> ❌ よくある誤り：「PCAで情報は失われない」「主成分には必ずはっきりした意味がある」。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 次元削減 | たくさんの変数を、少数の軸（主成分）にまとめる |
| 主成分 | ばらつき（情報）が最大の方向に引いた新しい軸 |
| 寄与率 | 各主成分が説明する情報の割合（第1主成分61%・累積で74%） |
| 負荷と命名 | 各変数の効き方を見て、人間が軸に意味を与える（例：規律） |
| ❌ 誤り | PCAで情報は失われない／主成分には必ず明確な意味がある |

> **PCAは『たくさんの変数の背後にある少数の軸』を見つける道具。**
> ただし情報は一部失われ、軸の意味づけは人間の解釈。

**課題（Moodle）**：PCAの寄与率と負荷を読み、第1・第2主成分が「何を表す軸か」を命名する。